## Core Concepts Cheat Sheet
- Architecture: Raw→Curated→Serving zones; Parquet/Avro; streaming (Pub/Sub+Dataflow) + batch (Dataproc/Spark); governance with Dataplex/Data Catalog; analytics in BigQuery.
- Performance: Spark—partitioning, shuffles, broadcast, AQE, small-file compaction; BigQuery—partitioning, clustering, materialized views, slots.
- Security: IAM least privilege; ABAC via IAM Conditions; BigQuery policy tags (column-level), row access policies; CMEK (KMS); VPC-SC; DLP.
- SQL: Windows (`ROW_NUMBER`, `RANK`, `LAG/LEAD`), CTEs, joins (anti/semi), aggregates and percentages; arrays/structs with `UNNEST` (BigQuery).

## Section A — Architecture & Modeling (4 questions)

A1. Minimal GCP Lakehouse
- Design a simple lakehouse for analytics-only workloads: define zones, storage formats, catalog/governance, and query layer.

A2. Streaming Ingestion & Replay
- Ingest real-time events at scale. How do you ensure idempotency, reprocessing, and dead-letter handling?

A3. Schema Evolution
- A new optional field is added to events. Describe compatible schema evolution and downstream resilience.

A4. Cost vs Latency
- Choose between BigQuery SQL transformations and Dataproc Spark jobs for a daily heavy aggregation. Explain trade-offs.

### Section A — Solutions (Concise)
- A1: Raw (GCS, Parquet/Avro, partitioned) → Curated (Parquet/BigQuery tables) → Serving (BigQuery marts/views). Govern via Dataplex domains/assets; tag sensitivity in Data Catalog; prefer columnar formats; partition by time/entity.
- A2: Pub/Sub topics → Dataflow for stream ETL with windowing; idempotent writes (dedupe keys); DLQ topic for failures; replay from Pub/Sub retention; persistent raw in GCS enables backfill.
- A3: Allow nullable additions; default missing fields; downstream `SAFE` functions; views resilient to new fields; document schema in Catalog; backfill derived fields where feasible.
- A4: BigQuery is serverless, simpler ops, good for large scans with partitioning/clustering; Spark offers custom logic/control but adds ops cost. Consider latency SLAs, cost (slots vs cluster), team expertise.

## Section B — Processing & Performance (4 questions)

B1. Skewed Spark Join
- Diagnose and fix skew in a fact→dimension join causing OOM and long shuffles.

B2. Small Files
- Millions of 5–50KB Parquet files hurt performance. What is your compaction strategy and target sizes?

B3. BigQuery Pruning
- A query with `created_at` filters scans too much. How do you reduce bytes scanned and latency?

B4. Partitioning & Clustering
- Choose partition keys and clustering for a high-volume table; justify by access patterns.

### Section B — Solutions (Concise)
- B1: Broadcast small dim; salt hot keys; enable AQE (`spark.sql.adaptive.enabled=true`); pre-aggregate; tune `spark.sql.shuffle.partitions`; ensure sort-merge joins on partitioned, sorted data.
- B2: Compaction to ~128–512MB Parquet files via Dataproc/Dataflow; use `coalesce()` and write ordering; schedule periodic compaction; enforce partition standards.
- B3: Partition on `created_at` (ingestion/time) and cluster on `customer_id`/`region`; apply partition filters; avoid `SELECT *`; materialize heavy aggregates; inspect query plan.
- B4: Time-based partitioning for append-heavy facts; clustering by high-cardinality keys used in filters/joins (e.g., `customer_id`); validate against workload and cost.

## Section C — Governance & Security (3 questions)

C1. HR Salary Access
- Ensure HR sees raw salary; others see masked or aggregated metrics.

C2. Tokenization for Joinability
- Join on email without exposing it. Describe deterministic hashing and salt management.

C3. Perimeter & Encryption
- Prevent data exfiltration and control encryption keys. Which GCP features do you use?

### Section C — Solutions (Concise)
- C1: BigQuery policy tags on `salary` for HR group; authorized views with masked columns for others; row access policies for department; audit logs; CMEK for datasets.
- C2: `SHA256(salt || LOWER(email))` stored as `email_hash`; salt in Secret Manager/KMS; rotate by creating v2 hash and migrating; document re-identification policies.
- C3: VPC-SC for service perimeters; CMEK (Cloud KMS) for storage/BigQuery; private endpoints; strict IAM; DLP scans for detection; logging/alerts.

## Section D — SQL (6 questions)
Assume BigQuery tables:
- orders(order_id, customer_id, region, order_ts TIMESTAMP, order_amount NUMERIC)
- customers(customer_id, region, signup_ts TIMESTAMP)
- events(event_id, customer_id, event_ts TIMESTAMP, event_type STRING)

D1. Top 3 highest-spending customers per region last month.

D2. Monthly retention: customers with an event both this month and last month.

D3. Percent of total spend per region last month.

D4. 15-minute session counts per customer in the last 7 days.

D5. Customers with events but no orders in the last 90 days (anti-join).

D6. Ranking with ties: return top 3 per region using RANK and explain tie behavior.

### Section D — Solutions

D1:
```sql
WITH last_month AS (
  SELECT * FROM `project.dataset.orders`
  WHERE order_ts >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 MONTH)
    AND order_ts < DATE_TRUNC(CURRENT_DATE(), MONTH)
), spend AS (
  SELECT customer_id, region, SUM(order_amount) AS total_spend
  FROM last_month
  GROUP BY customer_id, region
)
SELECT region, customer_id, total_spend
FROM (
  SELECT region, customer_id, total_spend,
         ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_spend DESC) AS rn
  FROM spend
)
WHERE rn <= 3
ORDER BY region, total_spend DESC;
```

D2:
```sql
WITH this_month AS (
  SELECT DISTINCT customer_id
  FROM `project.dataset.events`
  WHERE event_ts >= DATE_TRUNC(CURRENT_DATE(), MONTH)
    AND event_ts < DATE_ADD(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 MONTH)
), last_month AS (
  SELECT DISTINCT customer_id
  FROM `project.dataset.events`
  WHERE event_ts >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 MONTH)
    AND event_ts < DATE_TRUNC(CURRENT_DATE(), MONTH)
)
SELECT COUNT(*) AS retained
FROM this_month t
JOIN last_month l USING (customer_id);
```

D3:
```sql
WITH lm AS (
  SELECT customer_id, region, order_amount
  FROM `project.dataset.orders`
  WHERE order_ts >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 MONTH)
    AND order_ts < DATE_TRUNC(CURRENT_DATE(), MONTH)
), spend AS (
  SELECT region, customer_id, SUM(order_amount) AS total_spend
  FROM lm
  GROUP BY region, customer_id
), totals AS (
  SELECT region, SUM(total_spend) AS region_total
  FROM spend
  GROUP BY region
)
SELECT s.region, s.customer_id, s.total_spend,
       SAFE_DIVIDE(s.total_spend, t.region_total) AS pct_of_region
FROM spend s
JOIN totals t USING (region)
ORDER BY s.region, pct_of_region DESC;
```

D4:
```sql
WITH e AS (
  SELECT customer_id, event_ts
  FROM `project.dataset.events`
  WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
), ordered AS (
  SELECT customer_id, event_ts,
         LAG(event_ts) OVER (PARTITION BY customer_id ORDER BY event_ts) AS prev_ts
  FROM e
)
SELECT customer_id,
       COUNTIF(prev_ts IS NULL OR TIMESTAMP_DIFF(event_ts, prev_ts, MINUTE) > 15) AS sessions_last_week
FROM ordered
GROUP BY customer_id;
```

D5:
```sql
WITH active_events AS (
  SELECT DISTINCT customer_id
  FROM `project.dataset.events`
  WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY)
), recent_orders AS (
  SELECT DISTINCT customer_id
  FROM `project.dataset.orders`
  WHERE order_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY)
)
SELECT customer_id
FROM active_events ae
LEFT JOIN recent_orders ro USING (customer_id)
WHERE ro.customer_id IS NULL;
```

D6 (RANK ties explanation):
```sql
WITH last_month AS (
  SELECT customer_id, region, SUM(order_amount) AS total_spend
  FROM `project.dataset.orders`
  WHERE order_ts >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 MONTH)
    AND order_ts < DATE_TRUNC(CURRENT_DATE(), MONTH)
  GROUP BY customer_id, region
)
SELECT region, customer_id, total_spend,
       RANK() OVER (PARTITION BY region ORDER BY total_spend DESC) AS rnk
FROM last_month
WHERE rnk <= 3
ORDER BY region, total_spend DESC;
```
Explanation: `RANK` includes ties (you may see >3 rows). Use `ROW_NUMBER` for strict top-3, or `DENSE_RANK` to compress rank gaps but still include ties.